In [8]:
# Cell 1: Install only audio processing and tracking libraries
%pip -q install dagshub dvc noisereduce librosa soundfile scipy tqdm pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [9]:
# Cell 3: Preprocessing Configuration
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from scipy.signal import butter, filtfilt, iirnotch
import noisereduce as nr
from tqdm import tqdm
from pathlib import Path
import tqdm as notebook_tqdm

# DSP & Segment Parameters
TARGET_SR = 4000         # Sufficient for lung sounds up to 2000 Hz (Nyquist limit)
LOWCUT_FREQ = 25.0       # Respiratory sound bandpass lowcut
HIGHCUT_FREQ = 1800.0    # Respiratory sound bandpass highcut
MAINS_FREQ = 50.0        # Power line hum frequency (Europe/Asia)
FIXED_DURATION = 2.5     # Standard respiratory phase length in seconds
TARGET_LENGTH = int(TARGET_SR * FIXED_DURATION) # 10,000 samples

# Fixed-size log-mel spectrogram output: (N_MELS, TARGET_FRAMES)
N_MELS = 128
N_FFT = 512
HOP_LENGTH = 128
FMAX = HIGHCUT_FREQ
TARGET_FRAMES = 1 + TARGET_LENGTH // HOP_LENGTH

BASE_DIR = Path("HF_Lung")
OUTPUT_DIR = Path("preprocessed_npy")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
# Cell 4: Parse & Filter Annotations
annotations_file = BASE_DIR / "annotations.csv"
df = pd.read_csv(annotations_file)

# Restrict exclusively to respiratory phase events (Inhale / Exhale)
phase_map = {"I": 1, "E": 0}  # 1: Inhale, 0: Exhale
df = df[df["label"].isin(phase_map.keys())].copy()
df["target"] = df["label"].map(phase_map)

# Resolve disk filepaths based on split assignment
def get_audio_path(row):
    folder = "train" if row["split"] == "train" else "test"
    path = BASE_DIR / folder / row["filename"]
    return path if path.exists() else None

df["filepath"] = df.apply(get_audio_path, axis=1)
df = df.dropna(subset=["filepath"]).reset_index(drop=True)

print(f"Total valid segments: {len(df)}")
print(df["label"].value_counts())

Total valid segments: 52444
label
I    34095
E    18349
Name: count, dtype: int64


In [11]:
# Cell 5: DSP Signal Cleaning Functions
def butter_bandpass(lowcut, highcut, sr, order=4):
    nyquist = 0.5 * sr
    low = max(lowcut / nyquist, 1e-4)
    high = min(highcut / nyquist, 0.999)
    b, a = butter(order, [low, high], btype='band')
    return b, a

def apply_dsp_cleaning(audio: np.ndarray, sr: int) -> np.ndarray:
    """
    Applies:
    1. 50 Hz Notch filter (and 100 Hz harmonic) to eliminate mains hum.
    2. 4th-order Butterworth bandpass (25 Hz - 1800 Hz).
    3. Spectral gating for background noise reduction.
    """
    # 1. Powerline hum notch filter (50 Hz and 100 Hz)
    for freq in [MAINS_FREQ, MAINS_FREQ * 2]:
        if freq < (sr / 2):
            b_notch, a_notch = iirnotch(w0=freq, Q=30, fs=sr)
            audio = filtfilt(b_notch, a_notch, audio)

    # 2. Bandpass filtering
    b, a = butter_bandpass(LOWCUT_FREQ, HIGHCUT_FREQ, sr, order=4)
    audio = filtfilt(b, a, audio)

    # 3. Spectral Gating (Non-stationary noise reduction)
    audio = nr.reduce_noise(
        y=audio,
        sr=sr,
        stationary=False,
        prop_decrease=0.75,
        time_mask_smooth_ms=64,
        n_fft=512,
        win_length=512,
        hop_length=128,
    )

    return audio

def pad_or_truncate(audio_segment: np.ndarray, target_samples: int) -> np.ndarray:
    """Ensures uniform dimension for batch .npy export."""
    seg_len = len(audio_segment)
    if seg_len < target_samples:
        pad_width = target_samples - seg_len
        return np.pad(audio_segment, (0, pad_width), mode='constant')
    return audio_segment[:target_samples]

def compute_log_mel_spectrogram(audio_segment: np.ndarray, sr: int) -> np.ndarray:
    """Convert a fixed-length cleaned segment to a stable-size log-mel spectrogram."""
    mel = librosa.feature.melspectrogram(
        y=audio_segment,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=LOWCUT_FREQ,
        fmax=FMAX,
        center=True,
        power=2.0,
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    return log_mel.astype(np.float32)

In [12]:
# Cell 6: Full Pipeline Execution with Cache Optimization
def run_preprocessing_pipeline(metadata_df: pd.DataFrame, max_samples: int = None):
    if max_samples:
        metadata_df = metadata_df.sample(n=min(max_samples, len(metadata_df)), random_state=42)

    grouped = metadata_df.groupby("filepath")

    train_x, train_y = [], []
    test_x, test_y = [], []
    processed_records = []

    for filepath, group in tqdm(grouped, desc="Processing Audio Files"):
        try:
            # Load full recording once per file
            audio, sr = librosa.load(filepath, sr=TARGET_SR, mono=True)

            # Apply DSP filtering to full stream to prevent edge transients on slices
            clean_audio = apply_dsp_cleaning(audio, sr)

            for _, row in group.iterrows():
                start_sample = int(row["start_time"] * sr)
                end_sample = int(row["end_time"] * sr)

                if start_sample >= len(clean_audio) or start_sample >= end_sample:
                    continue

                slice_raw = clean_audio[start_sample:end_sample]
                if len(slice_raw) == 0:
                    continue

                # Amplitude Peak Normalization
                max_amp = np.max(np.abs(slice_raw))
                if max_amp > 1e-6:
                    slice_norm = slice_raw / max_amp
                else:
                    slice_norm = slice_raw

                # Uniform waveform shape followed by log-mel conversion
                processed_slice = pad_or_truncate(slice_norm, TARGET_LENGTH)
                processed_slice = compute_log_mel_spectrogram(processed_slice, sr)

                if row["split"] == "train":
                    train_x.append(processed_slice)
                    train_y.append(row["target"])
                else:
                    test_x.append(processed_slice)
                    test_y.append(row["target"])

                processed_records.append({
                    "filename": row["filename"],
                    "split": row["split"],
                    "target": row["target"],
                    "duration": row["duration"]
                })
        except Exception as e:
            print(f"Skipping {filepath.name} due to read error: {e}")

    # Convert to contiguous NumPy arrays
    X_train = np.array(train_x, dtype=np.float32)
    y_train = np.array(train_y, dtype=np.int64)
    X_test = np.array(test_x, dtype=np.float32)
    y_test = np.array(test_y, dtype=np.int64)

    # Save to disk
    np.save(OUTPUT_DIR / "X_train.npy", X_train)
    np.save(OUTPUT_DIR / "y_train.npy", y_train)
    np.save(OUTPUT_DIR / "X_test.npy", X_test)
    np.save(OUTPUT_DIR / "y_test.npy", y_test)

    manifest_df = pd.DataFrame(processed_records)
    manifest_df.to_csv(OUTPUT_DIR / "processed_manifest.csv", index=False)

    print("\n--- Pipeline Complete ---")
    print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
    print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

# Run the complete dataset; pass an integer to max_samples only for a quick test.
run_preprocessing_pipeline(df, max_samples=None)

Processing Audio Files: 100%|██████████| 9499/9499 [43:04<00:00,  3.68it/s]



--- Pipeline Complete ---
X_train shape: (42824, 128, 79) | y_train shape: (42824,)
X_test shape:  (9620, 128, 79)  | y_test shape:  (9620,)


In [13]:
# Cell 7: Verify Fast Data Loading for Model Training
X_tr = np.load(OUTPUT_DIR / "X_train.npy", mmap_mode="r")
y_tr = np.load(OUTPUT_DIR / "y_train.npy")

print(f"Successfully verified .npy files.")
print(f"Sample array dtype: {X_tr.dtype}")
print(f"Spectrogram shape: {X_tr.shape[1:]}")
assert X_tr.ndim == 3 and X_tr.shape[1:] == (N_MELS, TARGET_FRAMES)
print(f"Inhale count: {np.sum(y_tr == 1)}, Exhale count: {np.sum(y_tr == 0)}")

Successfully verified .npy files.
Sample array dtype: float32
Spectrogram shape: (128, 79)
Inhale count: 27223, Exhale count: 15601
